In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import torch
import numpy as np
from amd.rocal.pipeline import Pipeline
import amd.rocal.fn as fn
import amd.rocal.types as types

import ctypes
import matplotlib.pyplot as plt
import matplotlib.patches as patches


In [12]:
class ROCALCOCOIterator(object):
    """
    COCO ROCAL iterator for pyTorch.
    Parameters
    ----------
    pipelines : list of amd.rocal.pipeline.Pipeline
                List of pipelines to use
    size : int
           Epoch size.
    """

    def __init__(self, pipelines, tensor_layout=types.NHWC, reverse_channels=False, multiplier=None, offset=None, tensor_dtype=types.FLOAT, device="cpu", display=False, num_anchors=8732):

        try:
            assert pipelines is not None, "Number of provided pipelines has to be at least 1"
        except Exception as ex:
            print(ex)
        self.loader = pipelines
        self.tensor_format = tensor_layout
        self.multiplier = multiplier if multiplier else [1.0, 1.0, 1.0]
        self.offset = offset if offset else [0.0, 0.0, 0.0]
        self.reverse_channels = reverse_channels
        self.tensor_dtype = tensor_dtype
        self.device = device
        self.device_id = self.loader._device_id
        self.bs = self.loader._batch_size
        self.output_memory_type = self.loader._output_memory_type
        self.num_anchors = num_anchors
        self.display = True

        # Image id of a batch of images
        self.image_id = np.zeros(self.bs, dtype="int32")
        # Count of labels/ bboxes in a batch
        self.bboxes_label_count = np.zeros(self.bs, dtype="int32")
        # Image sizes of a batch
        self.img_size = np.zeros((self.bs * 2), dtype="int32")

    def next(self):
        return self.__next__()

    def __next__(self):
        if self.loader.rocal_run() != 0:
            raise StopIteration
        else:
            self.output_tensor_list = self.loader.get_output_tensors()

        # From init

        self.lis = []  # Empty list for bboxes
        self.lis_lab = []  # Empty list of labels
        self.output_list = []
        for i in range(len(self.output_tensor_list)):
            self.dimensions = self.output_tensor_list[j].dimensions()
            self.torch_dtype = self.output_tensor_list[j].dtype()
            if self.device == "cpu":
                torch_device = torch.device('cpu')
                self.output = torch.empty(self.dimensions, dtype=getattr(torch, self.torch_dtype))
            else:
                torch_device = torch.device('cuda', self.device_id)
                self.output = torch.empty(self.dimensions, dtype=getattr(torch, self.torch_dtype), device=torch_gpu_device)
            self.output_tensor_list[i].copy_data(ctypes.c_void_p(self.output.data_ptr()), self.output_memory_type)
            self.output_list.append(self.output)

        # 1D labels & bboxes array
        bbox_cpu = self.loader.get_bounding_box_cords()
        # bbox_arr = torch.as_tensor(
        #     bbox_cpu, dtype=torch.float32, device=torch_device)
        label_cpu = self.loader.get_bounding_box_labels()
        # label_arr = torch.as_tensor(
        #     label_cpu, dtype=torch.int32, device=torch_device)
        pixelwiselabel_cpu = self.loader.get_pixelwise_labels()
        random_mask_pixel_cpu = self.loader.get_random_mask_pixel()

        # Image id of a batch of images
        self.loader.get_image_id(self.image_id)
        # Image sizes of a batch
        self.loader.get_img_sizes(self.img_size)

        return self.output_list, bbox_cpu, label_cpu, pixelwiselabel_cpu, random_mask_pixel_cpu

    def reset(self):
        self.loader.rocal_reset_loaders()

    def __iter__(self):
        return self


def draw_patches(img, device):
    if device == "cpu":
        image = img.detach().cpu().numpy()
    else:
        image = img.cpu().numpy()
    image = (image).astype('uint8')
    return image

In [ ]:
image_path = "/data/MIVisionX-data/rocal_data/coco/coco_10_img/train_10images_2017/"
annotation_path = "/data/MIVisionX-data/rocal_data/coco/coco_10_img/annotations/instances_train2017.json"
_rali_cpu = True
batch_size = 10
num_threads = 1
device_id = 0
random_seed = 11
crop_width = 640
crop_height = 480

local_rank = 0
world_size = 1

rali_device = 'cpu'
decoder_device = 'cpu'
device_memory_padding = 211025920 if decoder_device == 'mixed' else 0
host_memory_padding = 140544512 if decoder_device == 'mixed' else 0

print("*********************************************************************")

coco_train_pipeline = Pipeline(batch_size=batch_size, num_threads=num_threads,
                               device_id=device_id, seed=random_seed, rocal_cpu=_rali_cpu)

with coco_train_pipeline:
    jpegs, bboxes, labels = fn.readers.coco(
        annotations_file=annotation_path, pixelwise_masks=True, is_box_encoder=False, is_foreground=True)

    print("*********************** SHARD ID ************************", local_rank)
    print("*********************** NUM SHARDS **********************", world_size)
    images_decoded = fn.decoders.image(jpegs, file_root=image_path, output_type=types.RGB,
                                       shard_id=0, num_shards=1, random_shuffle=False, annotations_file=annotation_path)
    res_images = fn.resize(images_decoded, device=rali_device, resize_width=crop_width,
                           resize_height=crop_height, output_layout=types.NHWC, output_dtype=types.UINT8)
    images = fn.crop_mirror_normalize(images_decoded, device="cpu",
                                      crop=(crop_height, crop_width),
                                      mirror=0,
                                      output_layout=types.NHWC,
                                      output_dtype=types.FLOAT,
                                      mean=[0, 0, 0],
                                      std=[1, 1, 1])

    coco_train_pipeline.set_outputs(images)
coco_train_pipeline.build()
COCOIteratorPipeline = ROCALCOCOIterator(coco_train_pipeline)
cnt = 0
fig, ax = plt.subplots(dpi=160)
for epoch in range(1):
    print("+++++++++++++++++++++++++++++EPOCH+++++++++++++++++++++++++++++++++++++", epoch)
    for i, it in enumerate(COCOIteratorPipeline):
        for j in range(batch_size):
            print("************************************** j *************************************", j)
            labels_array = it[3]
            img = it[0][0][j]
            ax.imshow(draw_patches(img, "cpu"))

            image = it[0][0][j]
            mask = it[3][j]
            tot_pixel = crop_height*crop_width
            mask = [mask[i:i+crop_width] for i in range(0, tot_pixel, crop_width)]
            center_array = it[4]
            center = it[4][j]
            crop_h = np.full([batch_size], 200)
            crop_w = np.full([batch_size], 200)
            shape_array = np.stack([crop_h, crop_w], axis=1)
            shape = shape_array[j]
            anchor_array = center_array - shape_array//2
            anchor = anchor_array[j]
            fig, ax = plt.subplots(dpi=160)
            ax.imshow(draw_patches(img, "cpu"))
            ax.imshow(mask, cmap='jet', alpha=0.5)
            rect = patches.Rectangle((anchor[0], anchor[1]), width=shape[0], height=shape[1],
                                    linewidth=1, edgecolor='#76b900', facecolor='none')
            ax.add_patch(rect)
            ax.scatter(center[0], center[1], s=10, edgecolor='#76b900')
            plt.title('Original Image/Mask with random crop window and center')
            anchor_cpu = anchor
            shape_cpu = shape
            print(shape_cpu[0], "  ", shape_cpu[1])
            anchor_x = center[0] - (shape_cpu[0] // 2)
            anchor_y = center[1] - (shape_cpu[1] // 2)
            image = image.cpu()
            np_array = np.array(image)
            np_mask = np.array(mask)
            end_x = anchor_x+shape_cpu[0]
            end_y = anchor_y+shape_cpu[1]
            end_x = crop_width-1 if end_x >= crop_width else end_x
            end_y = crop_height-1 if end_y >= crop_height else end_y
            cropped_array = np_array[anchor_y:end_y, anchor_x:end_x, :]
            cropped_mask = np_mask[anchor_y:end_y, anchor_x:end_x]

            fig, ax = plt.subplots(dpi=160)
            cropped_array = (np_array).astype('uint8')
            print(np_mask.shape)

            ax.imshow(cropped_array)
            ax.imshow(cropped_mask, cmap='jet', alpha=0.5)
            plt.title('Cropped image')

            plt.show()
    COCOIteratorPipeline.reset()
print("*********************************************************************")